In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: ptbxl-metadata-audit
#| tbl-cap: PTB-XL metadata audit computed from the local cohort file.
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..")
ptb = pd.read_csv(ROOT / "data/ptb_xl/ptbxl_database.csv")

ptb_audit = pd.DataFrame({
    "quantity": [
        "records", "unique patients", "male-coded (sex=0)",
        "female-coded (sex=1)", "age missing", "age outside [0,120]",
        "fold-10 test records", "fold-10 unique patients"
    ],
    "value": [
        len(ptb), ptb.patient_id.nunique(), (ptb.sex == 0).sum(),
        (ptb.sex == 1).sum(), ptb.age.isna().sum(),
        (~ptb.age.between(0, 120)).sum(), (ptb.strat_fold == 10).sum(),
        ptb.loc[ptb.strat_fold == 10, "patient_id"].nunique()
    ]
})
ptb_audit

,quantity,value
0,records,21799
1,unique patients,18869
2,male-coded (sex=0),11354
3,female-coded (sex=1),10445
4,age missing,0
5,"age outside [0,120]",293
6,fold-10 test records,2198
7,fold-10 unique patients,1904


In [3]:
#| label: ptbxl-patient-overlap
fold_groups = {
    "train": set(ptb.loc[ptb.strat_fold <= 8, "patient_id"]),
    "validation": set(ptb.loc[ptb.strat_fold == 9, "patient_id"]),
    "test": set(ptb.loc[ptb.strat_fold == 10, "patient_id"]),
}
pd.DataFrame([
    {"pair": f"{a} ∩ {b}", "overlapping_patients": len(fold_groups[a] & fold_groups[b])}
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]
])

,pair,overlapping_patients
0,train ∩ validation,0
1,train ∩ test,0
2,validation ∩ test,0


In [4]:
#| label: ptbxl-demographic-diagnostic-eda
import ast

ptb["analysis_split"] = np.select(
    [ptb.strat_fold <= 8, ptb.strat_fold == 9, ptb.strat_fold == 10],
    ["train", "validation", "test"],
    default="unassigned",
)
ptb["age_valid"] = ptb.age.where(ptb.age.between(0, 120))
ptb_demographics = ptb.groupby("analysis_split").agg(
    records=("ecg_id", "size"),
    patients=("patient_id", "nunique"),
    valid_age=("age_valid", "count"),
    age_median=("age_valid", "median"),
    age_q1=("age_valid", lambda x: x.quantile(0.25)),
    age_q3=("age_valid", lambda x: x.quantile(0.75)),
    male_coded_records=("sex", lambda x: (x == 0).sum()),
    female_coded_records=("sex", lambda x: (x == 1).sum()),
)

scp = pd.read_csv(ROOT / "data/ptb_xl/scp_statements.csv", index_col=0)
diagnostic_map = scp.loc[
    scp.diagnostic == 1, "diagnostic_class"
].dropna().to_dict()
superclass_rows = []
for row in ptb[["analysis_split", "scp_codes"]].itertuples(index=False):
    codes = ast.literal_eval(row.scp_codes)
    classes = sorted({
        diagnostic_map[code] for code in codes if code in diagnostic_map
    })
    superclass_rows.extend(
        {"split": row.analysis_split, "superclass": label}
        for label in classes
    )
superclass_counts = (
    pd.DataFrame(superclass_rows)
      .groupby(["split", "superclass"]).size()
      .rename("records").reset_index()
)
print(f"Metadata source: {(ROOT / 'data/ptb_xl/ptbxl_database.csv').resolve()}")
print("Split-level demographics")
display(ptb_demographics)
print("Multilabel diagnostic-superclass counts")
superclass_counts

Metadata source: /home/mithunmanivannan/projects/benchmarking_loss_functions_ecg_reconstruction/data/ptb_xl/ptbxl_database.csv
Split-level demographics


,records,patients,valid_age,age_median,age_q1,age_q3,male_coded_records,female_coded_records
analysis_split,,,,,,,,
test,2198,1904,2164,63.0,50.0,74.0,1132,1066
train,17418,15023,17201,61.0,50.0,72.0,9089,8329
validation,2183,1942,2141,62.0,49.0,73.0,1133,1050


Multilabel diagnostic-superclass counts


,split,superclass,records
0,test,CD,496
1,test,HYP,262
2,test,MI,550
3,test,NORM,963
4,test,STTC,521
5,train,CD,3907
6,train,HYP,2119
7,train,MI,4379
8,train,NORM,7596
9,train,STTC,4186


In [5]:
#| label: ptbxl-waveform-audit
#| tbl-cap: Lead-wise audit of real PTB-XL WFDB waveforms (fixed 250-record stratified sample).
import wfdb
from scipy.signal import resample_poly, welch

PTB_ROOT = (ROOT / "data/ptb_xl").resolve()
audit_rows = (
    ptb.groupby("strat_fold", group_keys=False)
       .sample(n=25, random_state=20260731)
       .sort_values(["strat_fold", "ecg_id"])
)
signals_hr = []
signals_lr_up = []
for row in audit_rows[["filename_hr", "filename_lr"]].itertuples(index=False):
    signal_hr, fields_hr = wfdb.rdsamp(str(PTB_ROOT / row.filename_hr))
    signal_lr, fields_lr = wfdb.rdsamp(str(PTB_ROOT / row.filename_lr))
    if fields_hr["fs"] != 500 or signal_hr.shape != (5000, 12):
        raise ValueError(f"Unexpected PTB-XL high-resolution contract: {row.filename_hr}")
    if fields_lr["fs"] != 100 or signal_lr.shape != (1000, 12):
        raise ValueError(f"Unexpected PTB-XL low-resolution contract: {row.filename_lr}")
    if fields_hr["sig_name"] != fields_lr["sig_name"]:
        raise ValueError(f"Lead-order mismatch: {row.filename_hr} vs {row.filename_lr}")
    signals_hr.append(signal_hr.astype(np.float32))
    signals_lr_up.append(
        resample_poly(signal_lr, up=5, down=1, axis=0).astype(np.float32)
    )
ptb_wave = np.stack(signals_hr)  # records × 5000 samples × leads
ptb_lr_up = np.stack(signals_lr_up)
fields = fields_hr
lead_names = fields["sig_name"]

finite = np.isfinite(ptb_wave)
lead_sd = ptb_wave.std(axis=1)
freq, psd = welch(ptb_wave, fs=500, axis=1, nperseg=1024)
band_power = {
    "baseline_<0.5Hz": np.trapz(psd[:, (freq < 0.5), :], freq[freq < 0.5], axis=1),
    "cardiac_0.5-40Hz": np.trapz(
        psd[:, (freq >= 0.5) & (freq <= 40), :],
        freq[(freq >= 0.5) & (freq <= 40)], axis=1
    ),
    "high_40-150Hz": np.trapz(
        psd[:, (freq > 40) & (freq <= 150), :],
        freq[(freq > 40) & (freq <= 150)], axis=1
    ),
}

ptb_signal_audit = pd.DataFrame({
    "lead": lead_names,
    "finite_percent": 100 * finite.mean(axis=(0, 1)),
    "p01_mV": np.quantile(ptb_wave, 0.01, axis=(0, 1)),
    "median_mV": np.median(ptb_wave, axis=(0, 1)),
    "p99_mV": np.quantile(ptb_wave, 0.99, axis=(0, 1)),
    "median_RMS_mV": np.median(np.sqrt(np.mean(ptb_wave ** 2, axis=1)), axis=0),
    "near_flat_records_percent": 100 * (lead_sd < 1e-4).mean(axis=0),
    "median_high_frequency_fraction": np.median(
        band_power["high_40-150Hz"] /
        np.maximum(
            band_power["cardiac_0.5-40Hz"] + band_power["high_40-150Hz"],
            1e-12
        ),
        axis=0
    ),
})
print(f"Loaded {len(ptb_wave)} paired 500 Hz/100 Hz real records from {PTB_ROOT}")
ptb_signal_audit

/tmp/ipykernel_2562293/2935539881.py:34: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2562293/2935539881.py:35: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2562293/2935539881.py:39: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



Loaded 250 paired 500 Hz/100 Hz real records from /home/mithunmanivannan/projects/benchmarking_loss_functions_ecg_reconstruction/data/ptb_xl


,lead,finite_percent,p01_mV,median_mV,p99_mV,median_RMS_mV,near_flat_records_percent,median_high_frequency_fraction
0,I,100.0,-0.255,-0.025,0.717,0.135444,0.0,0.007053
1,II,100.0,-0.338,-0.023,0.631,0.128146,0.0,0.009037
2,III,100.0,-0.572,0.007,0.388,0.103978,0.0,0.018296
3,AVR,100.0,-0.627,0.025,0.222,0.123415,0.0,0.006548
4,AVL,100.0,-0.270,-0.015,0.590,0.098433,0.0,0.012432
5,AVF,100.0,-0.385,-0.006,0.428,0.099144,0.0,0.015300
6,V1,100.0,-0.955,0.020,0.415,0.164051,0.0,0.002987
7,V2,100.0,-1.378,-0.001,0.719,0.243837,0.0,0.004114
8,V3,100.0,-1.168,-0.020,0.867,0.245921,0.0,0.006189
9,V4,100.0,-0.759,-0.035,1.145,0.238608,0.0,0.007089


In [6]:
#| label: ptbxl-cross-lead-audit
#| tbl-cap: Record-level PTB-XL cross-lead and lead-law diagnostics.
record_corr = np.stack([np.corrcoef(record, rowvar=False) for record in ptb_wave])
offdiag = ~np.eye(12, dtype=bool)
law_residuals = {
    "III-(II-I)": ptb_wave[:, :, 2] - (ptb_wave[:, :, 1] - ptb_wave[:, :, 0]),
    "aVR+(I+II)/2": ptb_wave[:, :, 3] + (ptb_wave[:, :, 0] + ptb_wave[:, :, 1]) / 2,
    "aVL-(I-II/2)": ptb_wave[:, :, 4] - (ptb_wave[:, :, 0] - ptb_wave[:, :, 1] / 2),
    "aVF-(II-I/2)": ptb_wave[:, :, 5] - (ptb_wave[:, :, 1] - ptb_wave[:, :, 0] / 2),
}
pd.DataFrame({
    "quantity": [
        "median record mean |off-diagonal correlation|",
        "99th percentile record condition number",
        *[f"median RMSE {name} (mV)" for name in law_residuals],
    ],
    "value": [
        np.median(np.abs(record_corr[:, offdiag]).mean(axis=1)),
        np.quantile([np.linalg.cond(c) for c in record_corr], 0.99),
        *[np.median(np.sqrt(np.mean(residual ** 2, axis=1)))
          for residual in law_residuals.values()],
    ],
})

,quantity,value
0,median record mean |off-diagonal correlation|,5.764277e-01
1,99th percentile record condition number,1.949073e+07
2,median RMSE III-(II-I) (mV),3.659228e-04
3,median RMSE aVR+(I+II)/2 (mV),4.359759e-04
4,median RMSE aVL-(I-II/2) (mV),4.764976e-04
5,median RMSE aVF-(II-I/2) (mV),4.799740e-04


In [7]:
#| label: ptbxl-split-resampling-audit
#| tbl-cap: Split-specific PTB-XL signal summaries and paired 100→500 Hz resampling fidelity.
record_rms = np.sqrt(np.mean(ptb_wave ** 2, axis=(1, 2)))
record_peak = np.max(np.abs(ptb_wave), axis=(1, 2))
record_flat_leads = (ptb_wave.std(axis=1) < 1e-4).sum(axis=1)
resampling_mse = np.mean((ptb_lr_up - ptb_wave) ** 2, axis=(1, 2))
resampling_corr = np.asarray([
    np.corrcoef(high.flatten(), upsampled.flatten())[0, 1]
    for high, upsampled in zip(ptb_wave, ptb_lr_up)
])
record_summary = pd.DataFrame({
    "split": audit_rows.analysis_split.to_numpy(),
    "record_rms_mV": record_rms,
    "record_peak_mV": record_peak,
    "near_flat_leads": record_flat_leads,
    "100_to_500_mse_mV2": resampling_mse,
    "100_to_500_pearson": resampling_corr,
})
record_summary.groupby("split").agg(
    sampled_records=("record_rms_mV", "size"),
    median_RMS_mV=("record_rms_mV", "median"),
    p99_peak_mV=("record_peak_mV", lambda x: x.quantile(0.99)),
    records_with_near_flat_lead=("near_flat_leads", lambda x: (x > 0).sum()),
    median_resampling_MSE_mV2=("100_to_500_mse_mV2", "median"),
    median_resampling_Pearson=("100_to_500_pearson", "median"),
)

,sampled_records,median_RMS_mV,p99_peak_mV,records_with_near_flat_lead,median_resampling_MSE_mV2,median_resampling_Pearson
split,,,,,,
test,25,0.198486,3.20880,0,0.000711,0.992421
train,200,0.190092,4.12157,0,0.000589,0.992119
validation,25,0.213631,5.64016,0,0.000559,0.992400


In [8]:
#| label: echonext-schema-audit
import json

echo = pd.read_csv(ROOT / "data/echonext/echonext_metadata_100k.csv")
wave = np.load(ROOT / "data/echonext/EchoNext_test_waveforms.npy", mmap_mode="r")
provenance = json.loads((ROOT / "data/echonext/PROVENANCE.json").read_text())

pd.DataFrame({
    "quantity": ["metadata rows", "metadata columns", "test records",
                 "samples per lead", "leads", "sampling rate (Hz)",
                 "duration (s)", "stored dtype"],
    "value": [len(echo), echo.shape[1], wave.shape[0], wave.shape[2],
              wave.shape[3], provenance["sampling_rate_hz"],
              wave.shape[2] / provenance["sampling_rate_hz"], str(wave.dtype)]
})

,quantity,value
0,metadata rows,100000
1,metadata columns,39
2,test records,5442
3,samples per lead,2500
4,leads,12
5,sampling rate (Hz),250.0
6,duration (s),10.0
7,stored dtype,float64


In [9]:
#| label: echonext-patient-overlap
patient_column = next(
    name for name in ["patient_key", "patient_id", "PatientID", "mrn"]
    if name in echo.columns
)
split_sets = {
    split: set(group[patient_column].dropna())
    for split, group in echo.groupby("split")
}
pd.DataFrame([
    {
        "split_a": a,
        "split_b": b,
        "unique_a": len(split_sets[a]),
        "unique_b": len(split_sets[b]),
        "overlapping_patients": len(split_sets[a] & split_sets[b]),
    }
    for a, b in [
        ("train", "val"), ("train", "test"),
        ("val", "test"), ("no_split", "train"),
        ("no_split", "val"), ("no_split", "test"),
    ]
])

,split_a,split_b,unique_a,unique_b,overlapping_patients
0,train,val,26218,4626,0
1,train,test,26218,5442,0
2,val,test,4626,5442,0
3,no_split,train,4618,26218,0
4,no_split,val,4618,4626,2119
5,no_split,test,4618,5442,2499


In [10]:
#| label: echonext-demographic-eda
echo_demographics = echo.groupby("split").agg(
    records=("ecg_key", "size"),
    patients=("patient_key", "nunique"),
    age_median=("age_at_ecg", "median"),
    age_q1=("age_at_ecg", lambda x: x.quantile(0.25)),
    age_q3=("age_at_ecg", lambda x: x.quantile(0.75)),
    acquisition_year_min=("acquisition_year", "min"),
    acquisition_year_max=("acquisition_year", "max"),
)
sex_by_split = (
    echo.groupby(["split", "sex"]).size()
        .rename("records").reset_index()
)
setting_by_split = (
    echo.groupby(["split", "location_setting"]).size()
        .rename("records").reset_index()
        .sort_values(["split", "records"], ascending=[True, False])
)
race_by_split = (
    echo.groupby(["split", "race_ethnicity"]).size()
        .rename("records").reset_index()
        .sort_values(["split", "records"], ascending=[True, False])
)
print("Split-level record/patient and age/year summary")
display(echo_demographics)
print("Sex counts")
display(sex_by_split)
print("Care-setting counts")
display(setting_by_split)
print("Race/ethnicity counts")
race_by_split

Split-level record/patient and age/year summary


,records,patients,age_median,age_q1,age_q3,acquisition_year_min,acquisition_year_max
split,,,,,,,
no_split,17457,4618,62.0,50.0,71.0,2008,2022
test,5442,5442,64.0,52.0,74.0,2008,2022
train,72475,26218,63.0,52.0,73.0,2008,2022
val,4626,4626,64.0,52.0,75.0,2008,2022


Sex counts


,split,sex,records
0,no_split,female,7808
1,no_split,male,9649
2,test,female,2731
3,test,male,2711
4,train,female,33524
5,train,male,38951
6,val,female,2356
7,val,male,2270


Care-setting counts


,split,location_setting,records
1,no_split,inpatient,9150
0,no_split,emergency,4983
2,no_split,outpatient,2854
3,no_split,procedural,470
5,test,inpatient,2203
4,test,emergency,1971
6,test,outpatient,1059
7,test,procedural,209
9,train,inpatient,34906
8,train,emergency,22811


Race/ethnicity counts


,split,race_ethnicity,records
2,no_split,hispanic,5207
5,no_split,white,4968
1,no_split,black,3328
4,no_split,unknown,2095
3,no_split,other,1320
0,no_split,asian,539
8,test,hispanic,1649
11,test,white,1569
7,test,black,846
10,test,unknown,768


In [11]:
#| label: echonext-missingness
#| tbl-cap: Highest EchoNext metadata missingness rates.
missing = (echo.isna().mean() * 100).sort_values(ascending=False).head(15)
missing.rename_axis("field").reset_index(name="missing_percent")

,field,missing_percent
0,tr_max_velocity_value,54.996
1,pasp_value,43.424
2,pericardial_effusion_value,11.823
3,pr_interval,10.369
4,ivs_measurement,9.133
5,lvpw_measurement,9.125
6,rv_systolic_function_value,9.096
7,tricuspid_regurgitation_value,9.036
8,aortic_stenosis_value,9.003
9,aortic_regurgitation_value,9.003


In [12]:
#| label: echonext-label-support
#| tbl-cap: EchoNext test-set endpoint support from the locked label audit.
label_audit = json.loads(
    (ROOT / "results/comprehensive_latest_48_models/raw/factorial_v4_2x4/"
            "echonext_shd_label_audit.json").read_text()
)
support = pd.DataFrame(label_audit["label_summary"]).T
support["prevalence_percent"] = 100 * support["positive"] / support["nonmissing"]
support.sort_values("prevalence_percent")

,nonmissing,positive,negative,prevalence_percent
pulmonary_regurgitation_moderate_or_greater_flag,5442,20,5422,0.367512
aortic_regurgitation_moderate_or_greater_flag,5442,66,5376,1.212789
pericardial_effusion_moderate_large_flag,5442,69,5373,1.267916
aortic_stenosis_moderate_or_greater_flag,5442,286,5156,5.255421
mitral_regurgitation_moderate_or_greater_flag,5442,337,5105,6.192576
tricuspid_regurgitation_moderate_or_greater_flag,5442,353,5089,6.486586
tr_max_gte_32_flag,5442,375,5067,6.890849
rv_systolic_dysfunction_moderate_or_greater_flag,5442,419,5023,7.699375
pasp_gte_45_flag,5442,699,4743,12.844542
lvef_lte_45_flag,5442,962,4480,17.677325


In [13]:
#| label: echonext-endpoint-cooccurrence
#| tbl-cap: Strongest pairwise EchoNext test-label overlaps (Jaccard index).
endpoint_columns = [
    column for column in echo.columns
    if column.endswith("_flag")
]
component_columns = [
    column for column in endpoint_columns
    if column != "shd_moderate_or_greater_flag"
]
echo_test = echo.loc[echo.split == "test"].copy()
labels = echo_test[component_columns].fillna(0).astype(bool)
burden = labels.sum(axis=1)
burden_table = (
    burden.value_counts().sort_index()
          .rename_axis("positive_component_labels")
          .reset_index(name="records")
)

cooccurrence = []
for i, left in enumerate(component_columns):
    for right in component_columns[i + 1:]:
        intersection = int((labels[left] & labels[right]).sum())
        union = int((labels[left] | labels[right]).sum())
        cooccurrence.append({
            "endpoint_a": left.removesuffix("_flag"),
            "endpoint_b": right.removesuffix("_flag"),
            "both_positive": intersection,
            "jaccard": intersection / union if union else np.nan,
        })
print("Number of positive component endpoints per test ECG")
display(burden_table)
pd.DataFrame(cooccurrence).sort_values(
    ["jaccard", "both_positive"], ascending=False
).head(20)

Number of positive component endpoints per test ECG


,positive_component_labels,records
0,0,3124
1,1,1147
2,2,541
3,3,312
4,4,171
5,5,96
6,6,41
7,7,8
8,8,2


,endpoint_a,endpoint_b,both_positive,jaccard
54,pasp_gte_45,tr_max_gte_32,350,0.483425
43,tricuspid_regurgitation_moderate_or_greater,pasp_gte_45,249,0.310087
44,tricuspid_regurgitation_moderate_or_greater,tr_max_gte_32,152,0.263889
6,lvef_lte_45,rv_systolic_dysfunction_moderate_or_greater,286,0.261187
50,rv_systolic_dysfunction_moderate_or_greater,pasp_gte_45,192,0.207343
41,tricuspid_regurgitation_moderate_or_greater,rv_systolic_dysfunction_moderate_or_greater,132,0.206250
3,lvef_lte_45,mitral_regurgitation_moderate_or_greater,196,0.177697
34,mitral_regurgitation_moderate_or_greater,tricuspid_regurgitation_moderate_or_greater,103,0.175468
8,lvef_lte_45,pasp_gte_45,246,0.173852
0,lvef_lte_45,lvwt_gte_13,279,0.159977


In [14]:
#| label: echonext-live-waveform-audit
rng = np.random.default_rng(20260731)
echo_idx = np.sort(rng.choice(wave.shape[0], size=256, replace=False))
z = np.asarray(wave[echo_idx, 0, :, :], dtype=np.float64)
means = np.asarray(provenance["normalization"]["mean"], dtype=float)
stds = np.asarray(provenance["normalization"]["std"], dtype=float)
echo_uv = z * stds + means

echo_laws = {
    "III-(II-I)": echo_uv[:, :, 2] - (echo_uv[:, :, 1] - echo_uv[:, :, 0]),
    "aVR+(I+II)/2": echo_uv[:, :, 3] + (echo_uv[:, :, 0] + echo_uv[:, :, 1]) / 2,
    "aVL-(I-II/2)": echo_uv[:, :, 4] - (echo_uv[:, :, 0] - echo_uv[:, :, 1] / 2),
    "aVF-(II-I/2)": echo_uv[:, :, 5] - (echo_uv[:, :, 1] - echo_uv[:, :, 0] / 2),
}
echo_live_audit = pd.DataFrame({
    "quantity": [
        "resolved waveform path", "sampled records", "finite values (%)",
        "global 0.1st percentile (µV)", "global 99.9th percentile (µV)",
        *[f"median record RMSE {name} (µV)" for name in echo_laws],
    ],
    "value": [
        str((ROOT / "data/echonext/EchoNext_test_waveforms.npy").resolve()),
        len(echo_idx), 100 * np.isfinite(echo_uv).mean(),
        np.quantile(echo_uv, 0.001), np.quantile(echo_uv, 0.999),
        *[np.median(np.sqrt(np.mean(residual ** 2, axis=1)))
          for residual in echo_laws.values()],
    ],
})
echo_live_audit

,quantity,value
0,resolved waveform path,/home/mithunmanivannan/projects/benchmarking_l...
1,sampled records,256
2,finite values (%),100.0
3,global 0.1st percentile (µV),-342.0
4,global 99.9th percentile (µV),288.0
5,median record RMSE III-(II-I) (µV),2.275649
6,median record RMSE aVR+(I+II)/2 (µV),0.833575
7,median record RMSE aVL-(I-II/2) (µV),1.49382
8,median record RMSE aVF-(II-I/2) (µV),1.406066
